# XSAMSum Evaluation — ROUGE + BERTScore (Chinese only, per model)

This notebook evaluates Chinese summaries from a results CSV on the XSAMSum test set using the multilingual ROUGE toolkit and BERTScore.

The input CSV may contain results from multiple models, distinguished by the `model_name` column. Predictions are read from `generated_summary_zh` and references from `reference_summary_zh`. Metrics are computed **separately for each model**.

**Structure**
1. Setup: imports, config, helpers, load CSV
2. Chinese evaluation — ROUGE, then BERTScore (raw), per model, then combine & save

**Language flag note.** The multilingual ROUGE toolkit expects full language names (`"chinese"`). BERTScore expects ISO codes (`"zh"`).

In [ ]:
# Install dependencies
!pip install datasets bert-score pandas
!pip install pyonmttok jieba six nltk absl-py
!pip install git+https://github.com/csebuetnlp/xl-sum.git#subdirectory=multilingual_rouge_scoring

In [ ]:
!pip install nltk

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')      # often needed by the same code path
nltk.download('punkt_tab')  # NLTK 3.8.2+ split punkt into punkt_tab

## 1. Setup

In [ ]:
import os
import pandas as pd
from datasets import load_dataset
from rouge_score import rouge_scorer
from bert_score import BERTScorer

pd.set_option('display.max_colwidth', 120)

In [ ]:
# Config
DATASET_NAME = "XSAMSum"
INPUT_CSV = "results.csv"  # path to the CSV containing generated + reference summaries for one or more models

MODEL_COL = "model_name"
PRED_COL = "generated_summary_zh"
REF_COL = "reference_summary_zh"

# BERTScore config for Chinese (matches ClidSum protocol)
BERTSCORE_MODEL_ZH = "hfl/chinese-bert-wwm-ext"
BERTSCORE_NUM_LAYERS_ZH = 8

# Multilingual ROUGE toolkit expects the full language name, not an ISO code
ROUGE_LANG_ZH = "chinese"

# ClidSum paper uses R-1 / R-2 / R-L
ROUGE_TYPES = ["rouge1", "rouge2", "rougeL"]
TOP_N_WORST = 10

In [ ]:
# Helper functions
def load_predictions(path):
    """Load predictions from a text file, one summary per line."""
    with open(path, "r", encoding="utf-8") as f:
        return f.read().splitlines()


def compute_rouge(predictions, references, rouge_types, language):
    """Compute per-pair and corpus-level ROUGE F1 scores.

    `language` is the full language name expected by the multilingual ROUGE toolkit,
    e.g. "chinese" or "english" — not an ISO code.
    """
    scorer = rouge_scorer.RougeScorer(
        rouge_types=rouge_types,
        lang=language,
        use_stemmer=True,
    )

    pair_scores = []
    aggregated = {rt: 0.0 for rt in rouge_types}

    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)  # (reference, hypothesis)
        pair = {rt: round(scores[rt].fmeasure * 100, 2) for rt in rouge_types}
        pair_scores.append(pair)
        for rt in rouge_types:
            aggregated[rt] += scores[rt].fmeasure

    n = len(predictions)
    corpus_scores = {rt: round(aggregated[rt] / n * 100, 2) for rt in rouge_types}
    return corpus_scores, pair_scores


def compute_bertscore(
    predictions, references, model_type, lang,
    num_layers=None, rescale_with_baseline=False,
    batch_size=32, verbose=True,
):
    """Compute per-pair and corpus-level BERTScore F1.

    `lang` here is the ISO code ("zh", "en") that the bert_score library expects.
    """
    predictions = list(predictions)
    references = list(references)
    scorer_kwargs = dict(
        model_type=model_type,
        lang=lang,
        batch_size=batch_size,
        rescale_with_baseline=rescale_with_baseline,
    )
    if num_layers is not None:
        scorer_kwargs["num_layers"] = num_layers
    scorer = BERTScorer(**scorer_kwargs)

    # Fix OverflowError on long inputs
    scorer._tokenizer.model_max_length = 512

    P, R, F1 = scorer.score(
        predictions, references,
        verbose=verbose, batch_size=batch_size,
    )

    pair_scores = [round(F1[i].item() * 100, 2) for i in range(len(predictions))]
    corpus_scores = {"f1": round(F1.mean().item() * 100, 2)}
    return corpus_scores, pair_scores

In [ ]:
# Load the results CSV
df = pd.read_csv(INPUT_CSV)

required_cols = {MODEL_COL, PRED_COL, REF_COL}
missing = required_cols - set(df.columns)
assert not missing, f"Missing expected columns: {missing}"

# Drop rows with missing predictions/references
df = df.dropna(subset=[PRED_COL, REF_COL]).reset_index(drop=True)

model_names = df[MODEL_COL].unique().tolist()
print(f"Loaded {len(df)} rows for {len(model_names)} model(s): {model_names}")
df[MODEL_COL].value_counts()

## 2. Chinese evaluation

### 2a. Group data by model

In [ ]:
# Group rows by model_name; each group has its own predictions/references list
groups = {
    name: g.reset_index(drop=True)
    for name, g in df.groupby(MODEL_COL)
}

for name, g in groups.items():
    print(f"{name}: {len(g)} prediction/reference pairs")

### 2b. Chinese ROUGE (per model)

Multilingual ROUGE toolkit with `lang="chinese"` — this enables Chinese word segmentation (jieba) before n-gram counting. Passing an unrecognized language (e.g. `"zh"`) silently falls back to whitespace tokenization, which for Chinese means character-level matching and substantially inflated scores.

ROUGE is computed independently for each model's predictions against its references.

In [ ]:
rouge_corpus = {}
rouge_pairs = {}

for name, g in groups.items():
    corpus, pairs = compute_rouge(
        g[PRED_COL].tolist(), g[REF_COL].tolist(),
        rouge_types=ROUGE_TYPES, language=ROUGE_LANG_ZH,
    )
    rouge_corpus[name] = corpus
    rouge_pairs[name] = pairs
    print(f"[{name}] ROUGE-1: {corpus['rouge1']:.2f}  ROUGE-2: {corpus['rouge2']:.2f}  ROUGE-L: {corpus['rougeL']:.2f}")

### 2c. Chinese BERTScore (raw, per model)

Uses `hfl/chinese-bert-wwm-ext` with `num_layers=8`, matching the ClidSum evaluation protocol. Raw F1 only — the `bert_score` library doesn't ship a precomputed baseline for this model, so rescaling isn't available.

Computed independently for each model.

In [ ]:
bs_raw_corpus = {}
bs_raw_pairs = {}

for name, g in groups.items():
    corpus, pairs = compute_bertscore(
        g[PRED_COL].tolist(), g[REF_COL].tolist(),
        model_type=BERTSCORE_MODEL_ZH,
        lang="zh",
        num_layers=BERTSCORE_NUM_LAYERS_ZH,
        rescale_with_baseline=False,
        batch_size=32,
    )
    bs_raw_corpus[name] = corpus
    bs_raw_pairs[name] = pairs
    print(f"[{name}] BERTScore F1 (raw): {corpus['f1']}")

### 2d. Combine results, inspect, and save

In [ ]:
# Corpus-level scores: one row per model
corpus_rows = []
for name in groups:
    row = {MODEL_COL: name, "n": len(groups[name])}
    row.update(rouge_corpus[name])
    row["bs_f1_raw"] = bs_raw_corpus[name]["f1"]
    corpus_rows.append(row)

corpus_df = pd.DataFrame(corpus_rows)
print(f"Corpus Eval Scores (Chinese) of {DATASET_NAME}, by model")
corpus_df

In [ ]:
# Pair-level scores: concatenate across models, tagged by model_name
pair_dfs = []
for name, g in groups.items():
    cols = (
        {MODEL_COL: [name] * len(g),
         "reference": g[REF_COL].tolist(),
         "prediction": g[PRED_COL].tolist()}
        | pd.DataFrame(rouge_pairs[name]).to_dict("list")
        | {"bs_f1_raw": bs_raw_pairs[name]}
    )
    pair_dfs.append(pd.DataFrame(cols))

pair_results_df = pd.concat(pair_dfs, ignore_index=True)

for name in groups:
    sub = pair_results_df[pair_results_df[MODEL_COL] == name]
    print(f"── [{name}] BERTScore F1 / ROUGE-L distribution ──")
    print(sub[["bs_f1_raw", "rougeL"]].describe().round(2))
    print()

In [ ]:
for name in groups:
    sub = pair_results_df[pair_results_df[MODEL_COL] == name]
    worst = sub.nsmallest(TOP_N_WORST, "rougeL")
    print(f"── [{name}] Top {TOP_N_WORST} worst examples by ROUGE-L of {DATASET_NAME} ──")
    display(worst)

In [ ]:
corpus_df.to_csv(f"corpus_scores_zh_{DATASET_NAME}.csv", index=False, encoding="utf-8-sig")
pair_results_df.to_csv(f"pair_scores_zh_{DATASET_NAME}.csv", index=False, encoding="utf-8-sig")
print("Saved per-model Chinese scores to CSV.")